# Typical IP Problems: Be the Decision Maker
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 4 · explore during class, about 5 minutes per problem; the final challenge uses live market prices.*

> **How to use this notebook.** Run the **Setup** cell first. Then, for each problem:
> 1. read the **problem**: its data (costs, capacities, distances…) are **fixed**: they describe the situation;
> 2. play with the **explorer**: the ticks and menus are the **decision variables** (yes/no choices, who does what,
>    which bin, which next city). Change them and watch the objective and every constraint react
>    (✔ satisfied, ✘ violated). Try to find the best feasible plan, then press **Load the optimal plan**;
> 3. read the **formulation**, then run the **PuLP** cell: the same model solved in Python.

For each problem, ask yourself:
1. **Which constraint stops you** when you try to improve the objective?
2. Is the **obvious (greedy) choice** also the optimal one?
3. How many plans are there? Could you check them all if the problem were 10 times larger?

In [ ]:
# @title ⚙️ Setup: run this cell first (no need to read the code)
import importlib, subprocess, sys
import importlib.metadata
try:                                   # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)
import base64, io, itertools, json, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from scipy.optimize import milp, LinearConstraint, Bounds

PARTS = []      # HTML pieces of the current view, written into one HTML widget (no Output widget: VS Code safe)


def ip_optimum(c, A_ub=None, b_ub=None, A_eq=None, b_eq=None, ub=1, maximize=False):
    """Optimal 0/1 (or integer up to ub) solution of the IP (HiGHS), used for the "load the optimal plan" button."""
    c = np.asarray(c, float)
    cons = []
    if A_ub is not None: cons.append(LinearConstraint(A_ub, -np.inf, b_ub))
    if A_eq is not None: cons.append(LinearConstraint(A_eq, b_eq, b_eq))
    r = milp(-c if maximize else c, constraints=cons, integrality=np.ones(len(c)), bounds=Bounds(0, ub))
    return np.round(r.x), (-r.fun if maximize else r.fun)


def banner(ok, text, top=False):
    color, bg = ("#067429", "#eaf9ef") if ok else ("#bd0a0a", "#fff0f0")
    (lambda h: PARTS.insert(0, h) if top else PARTS.append(h))(f'<div style="border:2px solid {color};background:{bg};color:{color};padding:8px 12px;'
                 f'border-radius:8px;font-size:15px;margin:6px 0">{text}</div>')


def table(df):
    PARTS.append(df.to_html(index=False, border=1, escape=False,
                            float_format=lambda v: f"{0.0 if abs(v) < 5e-3 else v:,.2f}")
                 .replace('<table', '<table style="border-collapse:collapse;margin:0 0 8px 0;font-size:13px"', 1))


def show(fig):
    fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    PARTS.append(f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"/>')


def constraints(rows):
    """rows: (constraint, value of the left-hand side, sense, right-hand side). Shows a ✔/✘ table, returns the violated ones."""
    tol = 1e-4
    ok = [(v <= r + tol) if s == "≤" else (v >= r - tol) if s == "≥" else abs(v - r) <= tol for _, v, s, r in rows]
    table(pd.DataFrame({"Constraint": [n for n, *_ in rows], "Your plan": [v for _, v, _, _ in rows],
                        "": [s for _, _, s, _ in rows], "Limit (data)": [r for *_, r in rows],
                        "OK?": ["✔" if o else '<b style="color:#bd0a0a">✘</b>' for o in ok]}))
    return [n for (n, *_), o in zip(rows, ok) if not o]


def verdict(violated, z, z_best, what, maximize=False, fmt=lambda v: f"{v:,.2f}"):
    """Banner: feasible or not, value of the objective, comparison with the optimum."""
    if violated:
        banner(False, f"<b>INFEASIBLE plan</b>: it violates {', '.join(violated)}.<br>{what} = {fmt(z)}, but this "
                      f"plan is not allowed. Change the decisions until every constraint shows ✔.", top=True)
        return
    gap = (z_best - z) if maximize else (z - z_best)
    tail = ("<b>This is the optimal plan.</b>" if abs(gap) < 1e-3 * max(1, abs(z_best)) else
            f"The best feasible plan reaches {fmt(z_best)}: you are {fmt(abs(gap))} "
            f"{'below' if maximize else 'above'} it. Can you do better?")
    banner(True, f"<b>Feasible plan</b>: {what} = {fmt(z)}.<br>{tail}", top=True)


def tick(desc, value=False):
    return W.Checkbox(value=value, description=desc, indent=False, layout=W.Layout(width="320px"))


def choice(desc, options, value):
    return W.Dropdown(options=options, value=value, description=desc, style={"description_width": "110px"},
                      layout=W.Layout(width="280px"))


def explorer(groups, view, best, chart_beside_controls=False):
    """groups: {"Column title": {name: widget}}; best: {name: optimal value}. Redraws on every change."""
    controls = {k: w for g in groups.values() for k, w in g.items()}
    defaults = {k: w.value for k, w in controls.items()}
    cols = [W.VBox([W.HTML(f"<b>{title}</b>")] + list(g.values())) for title, g in groups.items()]
    out = W.HTML()

    def refresh(_=None):
        PARTS.clear()
        try:
            view(**{k: w.value for k, w in controls.items()})
        except Exception:
            plt.close("all")
            PARTS.append(f"<pre style='color:#bd0a0a'>{traceback.format_exc()}</pre>")
        # layout: verdict banner on top, then tables (left) and chart (right) side by side, so it fits one screen
        banners = [h for h in PARTS if h.startswith("<div")]
        figures = [h for h in PARTS if h.startswith("<img")]
        tables = [h for h in PARTS if h not in banners and h not in figures]
        figures = [h.replace("<img ", '<img style="width:100%;max-width:520px;height:auto" ', 1) for h in figures]
        out.value = ("".join(banners) +
                     '<div style="display:flex;flex-wrap:nowrap;gap:18px;align-items:flex-start">'
                     f'<div style="flex:0 0 auto">{"".join(tables)}</div>'
                     f'<div style="flex:1 1 280px;min-width:240px">{"".join(figures)}</div></div>')

    def load(values):
        for k, w in controls.items():
            w.unobserve(refresh, names="value")
            w.value = values(k)
            w.observe(refresh, names="value")
        refresh()

    b_opt = W.Button(description="Load the optimal plan", button_style="success", layout=W.Layout(width="200px"))
    b_zero = W.Button(description="Reset", layout=W.Layout(width="120px"))
    b_opt.on_click(lambda _: load(lambda k: best[k]))
    b_zero.on_click(lambda _: load(lambda k: defaults[k]))
    for w in controls.values():
        w.observe(refresh, names="value")
    refresh()
    display(W.VBox([W.HBox(cols), W.HBox([b_opt, b_zero]), out]))


_ANS = json.loads(base64.b64decode("eyJFMV9ucHYiOiA3NS4wLCAiRTFfbiI6IDQsICJFMl90aW1lIjogNDEuMCwgIkUzX2NvdiI6IDUsICJFM19jb3N0IjogNC41LCAiRTRfc3ViIjogMiwgIkU0X2xlbiI6IDIzLjB9"))


def check(key, value, tol=0.01):
    """Compare your answer with the expected one (relative tolerance 1%)."""
    if value is None:
        print(f"⏳ {key}: not answered yet"); return
    exp = _ANS[key]
    ok = abs(float(value) - exp) <= tol * max(1.0, abs(exp))
    print(f"✅ {key}: correct" if ok else f"❌ {key}: not yet")


print("Ready. Run each explorer cell below, then change the decisions.")

---
## 1 · Knapsack: which projects to fund
A research office has **\$35M** to invest and five candidate projects. Each project is **funded entirely or not at all**
(you cannot fund 40% of a project): the decision of each project is a **yes/no** variable.

| Project | Investment (\$M) | NPV (\$M) |
|---|---|---|
| A | 15 | 25 |
| B | 20 | 30 |
| C | 10 | 18 |
| D | 8 | 12 |
| E | 12 | 20 |

**Try this**
* Fund the projects with the best NPV per dollar first (greedy). Is the plan optimal?
* Find a plan that uses the whole \$35M. Does using all the budget guarantee the best NPV?
* Load the optimal plan: which project did the greedy rule leave out, and why does that pay off?

In [ ]:
# @title 🎒 Knapsack explorer: tick the projects you fund, watch budget and NPV
def _explorer():
    NAMES = ["A", "B", "C", "D", "E"]
    INV = np.array([15, 20, 10, 8, 12])                    # investment ($M, fixed data)
    NPV = np.array([25, 30, 18, 12, 20])                   # net present value ($M)
    BUDGET = 35
    x_best, z_best = ip_optimum(NPV, A_ub=[INV], b_ub=[BUDGET], maximize=True)

    def view(xA, xB, xC, xD, xE):
        x = np.array([xA, xB, xC, xD, xE], float)
        spend, npv = INV @ x, NPV @ x
        violated = constraints([("Budget ($M)", spend, "≤", BUDGET)])
        verdict(violated, npv, z_best, "total NPV", maximize=True, fmt=lambda v: f"${v:,.0f}M")
        table(pd.DataFrame({"Project": NAMES, "Funded": ["✔" if v else "" for v in x], "Investment ($M)": INV,
                            "NPV ($M)": NPV, "NPV per $": NPV / INV}))
        fig, ax = plt.subplots(figsize=(5, 1.7))
        left = 0
        for n, inv, v in zip(NAMES, INV, x):
            if v:
                ax.barh(0, inv, left=left, color="#1a63a9", edgecolor="white")
                ax.text(left + inv / 2, 0, n, color="white", ha="center", va="center", weight="bold")
                left += inv
        ax.axvline(BUDGET, color="black", ls="--", lw=1)
        ax.set_xlim(0, max(45, left * 1.05)); ax.set_yticks([])
        ax.set_xlabel("Budget used ($M); dashed line = limit")
        show(fig)

    explorer({"Your decisions: fund the project?": dict(xA=tick("A · invest 15, NPV 25"), xB=tick("B · invest 20, NPV 30"),
                                                         xC=tick("C · invest 10, NPV 18"), xD=tick("D · invest 8, NPV 12"),
                                                         xE=tick("E · invest 12, NPV 20"))},
             view, dict(zip(["xA", "xB", "xC", "xD", "xE"], map(bool, x_best))))


_explorer()

### Formulation
**Variables:** $x_j \in \{0,1\}$ = 1 if project $j$ is funded.

$$
\begin{aligned}
\max\ & z = 25x_A + 30x_B + 18x_C + 12x_D + 20x_E \\
\text{s.t. } & 15x_A + 20x_B + 10x_C + 8x_D + 12x_E \le 35 && \text{(budget)} \\
& x_j \in \{0,1\}
\end{aligned}
$$

**General form:** $\max \sum_j v_j x_j$ s.t. $\sum_j w_j x_j \le W$, $x_j \in \{0,1\}$.

In [ ]:
import pulp

projects = ["A", "B", "C", "D", "E"]
invest = {"A": 15, "B": 20, "C": 10, "D": 8, "E": 12}             # $M
npv = {"A": 25, "B": 30, "C": 18, "D": 12, "E": 20}               # $M
budget = 35

model = pulp.LpProblem("knapsack", pulp.LpMaximize)
x = {j: model.add_variable(f"x_{j}", cat="Binary") for j in projects}   # 1 = funded
model += pulp.lpSum(npv[j] * x[j] for j in projects)                    # total NPV
model += pulp.lpSum(invest[j] * x[j] for j in projects) <= budget, "budget"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· NPV = ${pulp.value(model.objective):.0f}M")
print("Funded:", [j for j in projects if x[j].value() > 0.5], "· invested:",
      sum(invest[j] for j in projects if x[j].value() > 0.5), "$M")

---
## 2 · Assignment: who does which project
Four employees must each take **exactly one** of four projects, and every project needs **exactly one** employee.
The table gives the **hours** each employee needs for each project.

| | Project 1 | Project 2 | Project 3 | Project 4 |
|---|---|---|---|---|
| Alice | 12 | 15 | 10 | 8 |
| Bob | 14 | 11 | 16 | 13 |
| Carol | 9 | 13 | 12 | 11 |
| David | 11 | 9 | 14 | 10 |

**Try this**
* Give every employee the project where they are fastest. Why is that plan infeasible?
* Find a feasible plan below 40 hours. How many plans exist in total? (4! = 24)
* Load the optimal plan: does everybody get their fastest project?

In [ ]:
# @title 👥 Assignment explorer: choose a project for each employee, watch the hours
def _explorer():
    EMP = ["Alice", "Bob", "Carol", "David"]
    PROJ = ["Project 1", "Project 2", "Project 3", "Project 4"]
    T = np.array([[12, 15, 10, 8], [14, 11, 16, 13], [9, 13, 12, 11], [11, 9, 14, 10]])    # hours (fixed data)
    perms = list(itertools.permutations(range(4)))
    p_best = min(perms, key=lambda p: sum(T[i][p[i]] for i in range(4)))
    z_best = sum(T[i][p_best[i]] for i in range(4))

    def view(Alice, Bob, Carol, David):
        chosen = [Alice, Bob, Carol, David]
        idx = [PROJ.index(c) for c in chosen]
        hours = [T[i][idx[i]] for i in range(4)]
        violated = constraints([(f"{p} has exactly one employee", chosen.count(p), "=", 1) for p in PROJ])
        verdict(violated, sum(hours), z_best, "total hours", fmt=lambda v: f"{v:,.0f} h")
        table(pd.DataFrame({"Employee": EMP, "Project": chosen, "Hours": hours}))
        fig, ax = plt.subplots(figsize=(4.6, 2.6))
        ax.imshow(T, cmap="Blues", vmin=0, vmax=30)
        for i in range(4):
            for j in range(4):
                ax.text(j, i, T[i][j], ha="center", va="center")
            ax.add_patch(plt.Rectangle((idx[i] - .5, i - .5), 1, 1, fill=False, ec="#bd0a0a", lw=3))
        ax.set_xticks(range(4), ["P1", "P2", "P3", "P4"]); ax.set_yticks(range(4), EMP)
        ax.set_title("Hours (red = your choice)", fontsize=10)
        show(fig)

    explorer({"Your decisions: project of each employee": {e: choice(e, PROJ, PROJ[i]) for i, e in enumerate(EMP)}},
             view, {e: PROJ[p_best[i]] for i, e in enumerate(EMP)})


_explorer()

### Formulation
**Variables:** $x_{ij} \in \{0,1\}$ = 1 if employee $i$ takes project $j$.

$$
\begin{aligned}
\min\ & z = \sum_{i}\sum_{j} c_{ij}\, x_{ij} \\
\text{s.t. } & \sum_{j} x_{ij} = 1 \quad \forall i && \text{(each employee, one project)} \\
& \sum_{i} x_{ij} = 1 \quad \forall j && \text{(each project, one employee)} \\
& x_{ij} \in \{0,1\}
\end{aligned}
$$

The constraint matrix of this problem is special: even if you drop the integrality ($0 \le x_{ij} \le 1$), the LP solution
is already integer. That is why assignment problems are **easy** while knapsack-like problems are not in general.

In [ ]:
import pulp

emp = ["Alice", "Bob", "Carol", "David"]
proj = ["Project 1", "Project 2", "Project 3", "Project 4"]
hours = {"Alice": [12, 15, 10, 8], "Bob": [14, 11, 16, 13], "Carol": [9, 13, 12, 11], "David": [11, 9, 14, 10]}

model = pulp.LpProblem("assignment", pulp.LpMinimize)
x = {(i, j): model.add_variable(f"x_{i}_{j}", cat="Binary") for i in emp for j in range(4)}   # 1 = i does project j
model += pulp.lpSum(hours[i][j] * x[i, j] for i in emp for j in range(4))                    # total hours
for i in emp:
    model += pulp.lpSum(x[i, j] for j in range(4)) == 1, f"employee_{i}"
for j in range(4):
    model += pulp.lpSum(x[i, j] for i in emp) == 1, f"project_{j}"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· total = {pulp.value(model.objective):.0f} hours")
for i in emp:
    j = next(j for j in range(4) if x[i, j].value() > 0.5)
    print(f"  {i:<6} → {proj[j]} ({hours[i][j]} h)")

---
## 3 · Bin packing: fewest containers
A shipping company must pack six items into containers of capacity **100**. The weights are
**40, 35, 30, 25, 20, 15** (items 1 to 6). Every item goes in **one** container; use as **few containers** as possible.

**Try this**
* The total weight is 165: what is the minimum number of containers you could ever need? (a lower bound)
* Start with one item per container (6 bins). Merge them until no container exceeds 100.
* Can you reach the lower bound? Load the optimal plan and check the load of each container.

In [ ]:
# @title 📦 Bin packing explorer: choose the container of each item, watch the loads
def _explorer():
    W_ = np.array([40, 35, 30, 25, 20, 15]); CAP = 100; NB = 6        # weights (fixed data), capacity, bins available
    # IP: y_b (bin b used) then x_ib (item i in bin b)
    n = NB + 6 * NB
    c = np.r_[np.ones(NB), np.zeros(6 * NB)]
    A_eq = np.zeros((6, n)); A_ub = np.zeros((NB, n))
    for i in range(6):
        A_eq[i, NB + i * NB: NB + (i + 1) * NB] = 1                   # each item in exactly one bin
    for b in range(NB):
        A_ub[b, b] = -CAP
        for i in range(6):
            A_ub[b, NB + i * NB + b] = W_[i]                          # load of bin b <= CAP * y_b
    sol, z_best = ip_optimum(c, A_ub=A_ub, b_ub=np.zeros(NB), A_eq=A_eq, b_eq=np.ones(6))
    where = [int(np.argmax(sol[NB + i * NB: NB + (i + 1) * NB])) + 1 for i in range(6)]

    def view(i1, i2, i3, i4, i5, i6):
        bins = [i1, i2, i3, i4, i5, i6]
        load = [sum(W_[i] for i in range(6) if bins[i] == b) for b in range(1, NB + 1)]
        violated = constraints([(f"Load of bin {b}", load[b - 1], "≤", CAP) for b in range(1, NB + 1)])
        verdict(violated, sum(1 for l in load if l > 0), z_best, "containers used", fmt=lambda v: f"{v:.0f}")
        table(pd.DataFrame({"Item": range(1, 7), "Weight": W_, "Container": bins}))
        fig, ax = plt.subplots(figsize=(5, 2.6))
        for b in range(1, NB + 1):
            bottom = 0
            for i in range(6):
                if bins[i] == b:
                    ax.bar(b, W_[i], bottom=bottom, color="#1a63a9", edgecolor="white")
                    ax.text(b, bottom + W_[i] / 2, f"item {i + 1}", color="white", ha="center", va="center", fontsize=8)
                    bottom += W_[i]
        ax.axhline(CAP, color="#bd0a0a", ls="--", lw=1)
        ax.set_xticks(range(1, NB + 1)); ax.set_ylim(0, 150); ax.set_xlabel("Container"); ax.set_ylabel("Load (capacity 100)")
        show(fig)

    explorer({"Your decisions: container of each item": {f"i{i + 1}": choice(f"Item {i + 1} (w={W_[i]})", [1, 2, 3, 4, 5, 6], i + 1)
                                                         for i in range(6)}},
             view, {f"i{i + 1}": where[i] for i in range(6)})


_explorer()

### Formulation
**Variables:** $y_b \in \{0,1\}$ = 1 if container $b$ is used; $x_{ib} \in \{0,1\}$ = 1 if item $i$ goes in container $b$.

$$
\begin{aligned}
\min\ & z = \sum_{b} y_b \\
\text{s.t. } & \sum_{b} x_{ib} = 1 \quad \forall i && \text{(each item in one container)} \\
& \sum_{i} w_i\, x_{ib} \le C\, y_b \quad \forall b && \text{(capacity, only if the container is used)} \\
& x_{ib},\, y_b \in \{0,1\}
\end{aligned}
$$

The link $w x \le C y$ is the classic way to say *"if I use something, I pay for it"* (a **fixed-charge** constraint).

In [ ]:
import pulp

w = {1: 40, 2: 35, 3: 30, 4: 25, 5: 20, 6: 15}                    # item weights
C, bins = 100, range(1, 7)                                        # capacity, containers available

model = pulp.LpProblem("bin_packing", pulp.LpMinimize)
y = {b: model.add_variable(f"y_{b}", cat="Binary") for b in bins}                       # container b used
x = {(i, b): model.add_variable(f"x_{i}_{b}", cat="Binary") for i in w for b in bins}   # item i in container b
model += pulp.lpSum(y[b] for b in bins)                                                 # containers used
for i in w:
    model += pulp.lpSum(x[i, b] for b in bins) == 1, f"item_{i}"
for b in bins:
    model += pulp.lpSum(w[i] * x[i, b] for i in w) <= C * y[b], f"capacity_{b}"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· containers = {pulp.value(model.objective):.0f}")
for b in bins:
    items = [i for i in w if x[i, b].value() > 0.5]
    if items:
        print(f"  container {b}: items {items} · load {sum(w[i] for i in items)}")

---
## 4 · Set covering: where to build fire stations
A city has six districts and five candidate fire stations. A station covers the districts listed and has a cost (in \$M).
**Every district must be covered by at least one station.**

| Station | Covers districts | Cost (\$M) |
|---|---|---|
| A | 1, 2, 3 | 2.5 |
| B | 2, 4, 5 | 3.0 |
| C | 1, 3, 4 | 2.8 |
| D | 4, 5, 6 | 2.2 |
| E | 3, 5, 6 | 2.0 |

**Try this**
* Build every station: everything is covered, but what does it cost? Remove stations one by one.
* Is the cheapest station (E) in the optimal plan? Is the station that covers the most districts?
* Find a plan with only 2 stations. How do the costs compare with the optimal plan?

In [ ]:
# @title 🚒 Set covering explorer: choose the stations, watch the coverage of every district
def _explorer():
    ST = ["A", "B", "C", "D", "E"]
    COV = {"A": {1, 2, 3}, "B": {2, 4, 5}, "C": {1, 3, 4}, "D": {4, 5, 6}, "E": {3, 5, 6}}     # districts covered (fixed data)
    COST = np.array([2.5, 3.0, 2.8, 2.2, 2.0])                                                  # $M
    M = np.array([[1 if d in COV[s] else 0 for s in ST] for d in range(1, 7)])                  # districts x stations
    x_best, z_best = ip_optimum(COST, A_ub=-M, b_ub=-np.ones(6))

    def view(xA, xB, xC, xD, xE):
        x = np.array([xA, xB, xC, xD, xE], float)
        count = M @ x
        violated = constraints([(f"District {d} covered by", count[d - 1], "≥", 1) for d in range(1, 7)])
        verdict(violated, COST @ x, z_best, "total cost", fmt=lambda v: f"${v:,.1f}M")
        table(pd.DataFrame({"Station": ST, "Built": ["✔" if v else "" for v in x], "Covers": [sorted(COV[s]) for s in ST],
                            "Cost ($M)": COST}))
        fig, ax = plt.subplots(figsize=(4.8, 2.4))
        grid = M * (1 + x[None, :])                       # 0 not covered by it, 1 covers (not built), 2 covers (built)
        ax.imshow(grid, cmap="Greens", vmin=0, vmax=2.2)
        ax.set_xticks(range(5), ST); ax.set_yticks(range(6), [f"D{d}" for d in range(1, 7)])
        for d in range(6):
            ax.text(5, d, "✔" if count[d] >= 1 else "✘", color="#067429" if count[d] >= 1 else "#bd0a0a",
                    ha="left", va="center", weight="bold")
        ax.set_xlim(-.5, 5.6); ax.set_title("Dark green = covered by a built station", fontsize=9)
        show(fig)

    explorer({"Your decisions: build the station?": dict(xA=tick("A · cost 2.5"), xB=tick("B · cost 3.0"), xC=tick("C · cost 2.8"),
                                                          xD=tick("D · cost 2.2"), xE=tick("E · cost 2.0"))},
             view, dict(zip(["xA", "xB", "xC", "xD", "xE"], map(bool, x_best))))


_explorer()

### Formulation
**Variables:** $x_j \in \{0,1\}$ = 1 if station $j$ is built.

$$
\begin{aligned}
\min\ & z = \sum_{j} c_j\, x_j \\
\text{s.t. } & \sum_{j \,:\, d \in \text{cover}(j)} x_j \ge 1 \quad \forall \text{ district } d && \text{(every district covered)} \\
& x_j \in \{0,1\}
\end{aligned}
$$

Notice the **$\ge 1$**: a district covered twice is allowed (just wasteful). Coverage constraints are the heart of
facility location and service design.

In [ ]:
import pulp

cover = {"A": [1, 2, 3], "B": [2, 4, 5], "C": [1, 3, 4], "D": [4, 5, 6], "E": [3, 5, 6]}   # districts covered
cost = {"A": 2.5, "B": 3.0, "C": 2.8, "D": 2.2, "E": 2.0}                                   # $M
districts = range(1, 7)

model = pulp.LpProblem("set_covering", pulp.LpMinimize)
x = {j: model.add_variable(f"x_{j}", cat="Binary") for j in cover}                          # 1 = station built
model += pulp.lpSum(cost[j] * x[j] for j in cover)                                          # total cost
for d in districts:
    model += pulp.lpSum(x[j] for j in cover if d in cover[j]) >= 1, f"district_{d}"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):.1f}M")
print("Stations built:", [j for j in cover if x[j].value() > 0.5])

---
## 5 · Traveling salesman: the shortest tour, and the subtour trap
A salesman starts in city **A**, visits **B, C, D** exactly once each and returns to A. The distances are symmetric:

| | A | B | C | D |
|---|---|---|---|---|
| A | – | 10 | 15 | 20 |
| B | 10 | – | 35 | 25 |
| C | 15 | 35 | – | 30 |
| D | 20 | 25 | 30 | – |

Here the decision is the **next city** after each city. The first constraints are the assignment ones (each city is
left once and entered once), but they are **not enough**: the plan may split into several small loops (**subtours**).

**Try this**
* Start from the tour A→B→C→D→A (95). Improve it by changing successors until every city is entered once.
* Set A→B, B→A, C→D, D→C: every city is left and entered once. Is it a tour? What is its length?
* Several plans have length 80. Which of them are real tours? Why does the model need extra constraints?

In [ ]:
# @title 🧭 TSP explorer: choose the next city after each city, watch the tour (or the subtours)
def _explorer():
    CITIES = ["A", "B", "C", "D"]
    D = np.array([[0, 10, 15, 20], [10, 0, 35, 25], [15, 35, 0, 30], [20, 25, 30, 0]])       # distances (fixed data)
    POS = {"A": (0, 2), "B": (3, 2), "C": (3, 0), "D": (0, 0)}
    tours = [(0,) + p for p in itertools.permutations(range(1, 4))]
    t_best = min(tours, key=lambda t: sum(D[t[k]][t[(k + 1) % 4]] for k in range(4)))
    z_best = sum(D[t_best[k]][t_best[(k + 1) % 4]] for k in range(4))
    nxt_best = {CITIES[t_best[k]]: CITIES[t_best[(k + 1) % 4]] for k in range(4)}

    def view(A, B, C, D_):
        succ = {"A": A, "B": B, "C": C, "D": D_}
        arrivals = {c: sum(1 for s in succ.values() if s == c) for c in CITIES}
        walk, cur = [], "A"
        while cur not in walk:                         # follow the successors from A until we come back
            walk.append(cur); cur = succ[cur]
        dist = sum(D[CITIES.index(a)][CITIES.index(b)] for a, b in succ.items())
        rows = [(f"Arrivals at {c}", arrivals[c], "=", 1) for c in CITIES] + [("Cities on the loop that starts at A", len(walk), "=", 4)]
        violated = constraints(rows)
        verdict(violated, dist, z_best, "tour length", fmt=lambda v: f"{v:,.0f}")
        if not violated:
            banner(True, "It is a <b>single tour</b>: " + " → ".join(walk + ["A"]))
        elif all(a == 1 for a in arrivals.values()):
            banner(False, "Every city is entered once, but the plan is split into <b>subtours</b>: the assignment constraints "
                          "do not forbid them.")
        fig, ax = plt.subplots(figsize=(4.4, 3.2))
        for c, (px, py) in POS.items():
            ax.scatter(px, py, s=600, color="#d9e6f5", edgecolor="#1a63a9", zorder=3)
            ax.text(px, py, c, ha="center", va="center", weight="bold", zorder=4)
        for a, b in succ.items():
            (x1, y1), (x2, y2) = POS[a], POS[b]
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1), zorder=2,
                        arrowprops=dict(arrowstyle="-|>", color="#bd0a0a", lw=2, shrinkA=16, shrinkB=16, connectionstyle="arc3,rad=0.12"))
        ax.set_xlim(-.7, 3.7); ax.set_ylim(-.7, 2.7); ax.axis("off")
        show(fig)

    explorer({"Your decisions: next city after each city":
              dict(A=choice("After A", ["B", "C", "D"], "B"), B=choice("After B", ["A", "C", "D"], "C"),
                   C=choice("After C", ["A", "B", "D"], "D"), D_=choice("After D", ["A", "B", "C"], "A"))},
             view, dict(A=nxt_best["A"], B=nxt_best["B"], C=nxt_best["C"], D_=nxt_best["D"]))


_explorer()

### Formulation (basic model)
**Variables:** $x_{ij} \in \{0,1\}$ = 1 if the salesman goes directly from city $i$ to city $j$.

$$
\begin{aligned}
\min\ & z = \sum_{i}\sum_{j \ne i} d_{ij}\, x_{ij} \\
\text{s.t. } & \sum_{j \ne i} x_{ij} = 1 \quad \forall i && \text{(leave each city once)} \\
& \sum_{i \ne j} x_{ij} = 1 \quad \forall j && \text{(enter each city once)} \\
& x_{ij} \in \{0,1\}
\end{aligned}
$$

This is an **assignment problem**: it can return several small loops. **Subtour elimination** constraints
(or the MTZ formulation) must be added to force a single tour: we will see them in the next sessions.

In [ ]:
import pulp

cities = ["A", "B", "C", "D"]
dist = {("A", "B"): 10, ("A", "C"): 15, ("A", "D"): 20, ("B", "C"): 35, ("B", "D"): 25, ("C", "D"): 30}
d = {**dist, **{(j, i): v for (i, j), v in dist.items()}}          # symmetric distances

model = pulp.LpProblem("tsp_basic", pulp.LpMinimize)
x = {(i, j): model.add_variable(f"x_{i}_{j}", cat="Binary") for i in cities for j in cities if i != j}
model += pulp.lpSum(d[i, j] * x[i, j] for (i, j) in x)                                   # total distance
for i in cities:
    model += pulp.lpSum(x[i, j] for j in cities if j != i) == 1, f"leave_{i}"
    model += pulp.lpSum(x[j, i] for j in cities if j != i) == 1, f"enter_{i}"
result = model.solve(pulp.COIN_CMD(msg=False))

succ = {i: j for (i, j) in x if x[i, j].value() > 0.5}
loops, seen = [], set()
for s in cities:                                                  # extract the loops of the solution
    if s not in seen:
        loop, cur = [], s
        while cur not in seen:
            seen.add(cur); loop.append(cur); cur = succ[cur]
        loops.append(loop)
print(result.status_str, f"· distance = {pulp.value(model.objective):.0f}")
print("Loops:", loops, "→ a single tour ✔" if len(loops) == 1 else "→ SUBTOURS: not a valid tour ✘")

---
## What did you learn?
* In an **integer program** the decisions are discrete (yes/no, which one, how many): you cannot "round" your way to
  the best plan, and the greedy choice is **not always optimal** (knapsack).
* The **number of plans explodes** (2⁵ projects, 4! assignments, (n−1)! tours…): that is why we need a solver
  that is smarter than checking every plan.
* **Structure matters**: assignment problems are easy (their LP is already integer), knapsack, bin packing, set covering
  and TSP are hard in general.
* Constraints encode the business rules: *exactly one* (`=`), *at most* (`≤`, resources), *at least* (`≥`, coverage),
  and *if I use it, I pay for it* (`w x ≤ C y`).
* A model can be **too permissive**: the basic TSP allows subtours, so the model must be strengthened.

---
## Your turn: modify a model and solve it
In each exercise, start from the PuLP code of the problem above, change what the situation requires, solve
it, and run the ✅ check. Write the model first (on paper): **what changes, the data or the structure?**

### Exercise 1 · Knapsack: a new project and a conflict
The office receives a larger budget of **\$45M** and a sixth project **F** (investment 14, NPV 21).
Projects **B and D are mutually exclusive** (they use the same team): at most one of them can be funded.
What is the best total NPV? How many projects are funded?

In [ ]:
import pulp

projects = ["A", "B", "C", "D", "E"]                              # ✍️ add "F"
invest = {"A": 15, "B": 20, "C": 10, "D": 8, "E": 12}             # ✍️ add the investment of F
npv = {"A": 25, "B": 30, "C": 18, "D": 12, "E": 20}               # ✍️ add the NPV of F
budget = 35                                                       # ✍️ new budget

model = pulp.LpProblem("knapsack_conflict", pulp.LpMaximize)
x = {j: model.add_variable(f"x_{j}", cat="Binary") for j in projects}
model += pulp.lpSum(npv[j] * x[j] for j in projects)
model += pulp.lpSum(invest[j] * x[j] for j in projects) <= budget, "budget"
# ✍️ add the conflict: B and D cannot both be funded
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· NPV = ${pulp.value(model.objective):.0f}M")
print("Funded:", [j for j in projects if x[j].value() > 0.5])
E1_npv = None           # ✍️ best total NPV ($M)
E1_n = None             # ✍️ number of projects funded
check("E1_npv", E1_npv); check("E1_n", E1_n)

### Exercise 2 · Assignment: a forbidden pair
**Bob cannot work on Project 2** (a conflict of interest). What is the new minimum total time?
Does the optimal plan change for the other employees too?

In [ ]:
import pulp

emp = ["Alice", "Bob", "Carol", "David"]
hours = {"Alice": [12, 15, 10, 8], "Bob": [14, 11, 16, 13], "Carol": [9, 13, 12, 11], "David": [11, 9, 14, 10]}

model = pulp.LpProblem("assignment_forbidden", pulp.LpMinimize)
x = {(i, j): model.add_variable(f"x_{i}_{j}", cat="Binary") for i in emp for j in range(4)}
model += pulp.lpSum(hours[i][j] * x[i, j] for i in emp for j in range(4))
for i in emp:
    model += pulp.lpSum(x[i, j] for j in range(4)) == 1, f"employee_{i}"
for j in range(4):
    model += pulp.lpSum(x[i, j] for i in emp) == 1, f"project_{j}"
# ✍️ forbid Bob on Project 2 (index 1)
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· total = {pulp.value(model.objective):.0f} hours")
for i in emp:
    print(f"  {i:<6} → Project {next(j for j in range(4) if x[i, j].value() > 0.5) + 1}")
E2_time = None          # ✍️ minimum total hours
check("E2_time", E2_time)

### Exercise 3 · Maximum coverage: not enough budget to cover everything
The city only has **\$4.6M**. It can no longer guarantee that every district is covered, so the goal changes:
**maximize the number of districts covered** without exceeding the budget.
You need a new binary variable $z_d$ = 1 if district $d$ is covered, with $z_d \le \sum_{j \text{ covers } d} x_j$.
How many districts can be covered, and how much does the chosen plan cost?

In [ ]:
import pulp

cover = {"A": [1, 2, 3], "B": [2, 4, 5], "C": [1, 3, 4], "D": [4, 5, 6], "E": [3, 5, 6]}
cost = {"A": 2.5, "B": 3.0, "C": 2.8, "D": 2.2, "E": 2.0}
districts = range(1, 7)
budget = 4.6

model = pulp.LpProblem("max_coverage", pulp.LpMaximize)
x = {j: model.add_variable(f"x_{j}", cat="Binary") for j in cover}              # station built
z = {d: model.add_variable(f"z_{d}", cat="Binary") for d in districts}          # district covered
# ✍️ objective: maximize the number of districts covered
# ✍️ budget constraint on the stations built
# ✍️ link: a district counts as covered only if one of its stations is built
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str)
print("Stations:", [j for j in cover if x[j].value() > 0.5], "· covered:", [d for d in districts if z[d].value() > 0.5])
E3_cov = None           # ✍️ number of districts covered
E3_cost = None          # ✍️ cost of the stations built ($M)
check("E3_cov", E3_cov); check("E3_cost", E3_cost)

### Exercise 4 · TSP: five cities and the subtour trap
Five cities at the coordinates below (distances rounded to one decimal, in the matrix `d`): two cities are close to each
other (0 and 1) and three form another cluster far away (2, 3 and 4).
1. Solve the **basic model** (assignment constraints only): how many loops does the solution have?
2. Add the **MTZ constraints** $u_i - u_j + n\,x_{ij} \le n-1$ for $i \ne j$, $i,j \ge 1$, with $1 \le u_i \le n-1$
   (the order in which city $i$ is visited): what is the length of the optimal **tour**?

In [ ]:
import pulp

xy = [(0, 0), (1, 0), (10, 0), (11, 1), (10, 2)]
n = len(xy)
d = {(i, j): round(((xy[i][0] - xy[j][0]) ** 2 + (xy[i][1] - xy[j][1]) ** 2) ** 0.5, 1) for i in range(n) for j in range(n)}

def solve_tsp(subtour_elimination):
    model = pulp.LpProblem("tsp5", pulp.LpMinimize)
    x = {(i, j): model.add_variable(f"x_{i}_{j}", cat="Binary") for i in range(n) for j in range(n) if i != j}
    u = {i: model.add_variable(f"u_{i}", lowBound=1, upBound=n - 1, cat="Continuous") for i in range(1, n)}   # visit order
    model += pulp.lpSum(d[i, j] * x[i, j] for (i, j) in x)
    for i in range(n):
        model += pulp.lpSum(x[i, j] for j in range(n) if j != i) == 1, f"leave_{i}"
        model += pulp.lpSum(x[j, i] for j in range(n) if j != i) == 1, f"enter_{i}"
    if subtour_elimination:
        pass                                                      # ✍️ add the MTZ constraints on u and x
    model.solve(pulp.COIN_CMD(msg=False))
    succ = {i: j for (i, j) in x if x[i, j].value() > 0.5}
    loops, seen = 0, set()
    for s in range(n):                                            # count the loops of the solution
        if s not in seen:
            loops += 1; cur = s
            while cur not in seen:
                seen.add(cur); cur = succ[cur]
    return pulp.value(model.objective), loops

length, loops = solve_tsp(False); print(f"basic model: length {length:.1f}, {loops} loop(s)")
length, loops = solve_tsp(True);  print(f"with MTZ:    length {length:.1f}, {loops} loop(s)")
E4_sub = None           # ✍️ number of loops of the basic model
E4_len = None           # ✍️ length of the optimal tour
check("E4_sub", E4_sub); check("E4_len", E4_len)

---
## Final challenge · Build a stock portfolio with live prices
You advise an investor who wants to **buy whole shares** of a few stocks from a predefined basket of **15 Dow Jones
stocks**. The prices come **live** from Yahoo Finance (`yfinance`): your model works with the **latest trade price**,
so the answer changes every day.

*This is a modeling exercise, not financial advice: past returns do not predict future returns.*

**The rules of the investor (data, they do not change)**

| Rule | Value |
|---|---|
| Budget to invest | \$100,000 (cash that is not invested stays cash) |
| Shares | whole shares only (integer ≥ 0) |
| Max. weight per stock | 25% of the budget (\$25,000) |
| Max. number of different stocks | 6 |
| Min. number of different sectors | 3 |
| Risk limit | value-weighted average volatility of the portfolio ≤ 21% |
| Goal | **maximize the expected 1-year profit**, estimated with each stock's return of the last 12 months |

In [ ]:
# @title 📈 Live market data: latest trade prices, 12-month return and volatility
%pip install -q yfinance
import yfinance as yf

INDEX = {"AAPL": ("Apple", "Technology"), "MSFT": ("Microsoft", "Technology"), "NVDA": ("Nvidia", "Technology"),
         "JPM": ("JPMorgan Chase", "Financials"), "GS": ("Goldman Sachs", "Financials"), "V": ("Visa", "Financials"),
         "JNJ": ("Johnson & Johnson", "Health care"), "UNH": ("UnitedHealth", "Health care"),
         "PG": ("Procter & Gamble", "Staples"), "KO": ("Coca-Cola", "Staples"), "WMT": ("Walmart", "Staples"),
         "MCD": ("McDonald's", "Discretionary"), "HD": ("Home Depot", "Discretionary"),
         "CAT": ("Caterpillar", "Industrials"), "CVX": ("Chevron", "Energy")}
# offline copy (2026-10-29), used only if Yahoo Finance cannot be reached: (price, 12-month return, annual volatility)
SNAPSHOT = {'AAPL': (349.74, 0.302, 0.241), 'MSFT': (523.86, -0.025, 0.329), 'NVDA': (213.14, 0.032, 0.372), 'JPM': (343.43, 0.146, 0.221), 'GS': (930.28, 0.211, 0.319), 'V': (409.87, 0.211, 0.223), 'JNJ': (274.58, 0.505, 0.207), 'UNH': (406.54, 0.175, 0.347), 'PG': (159.17, 0.101, 0.208), 'KO': (92.58, 0.391, 0.189), 'WMT': (121.14, 0.192, 0.263), 'MCD': (255.57, -0.133, 0.196), 'HD': (324.21, -0.118, 0.267), 'CAT': (656.11, 0.13, 0.398), 'CVX': (230.4, 0.544, 0.242)}

try:
    hist = yf.download(list(INDEX), period="1y", auto_adjust=True, progress=False, threads=False)["Close"].dropna(how="all").ffill()
    if hist.empty or hist.iloc[-1].isna().any():
        raise ValueError("incomplete data")
    price = hist.iloc[-1]                                          # latest trade price
    ret = hist.iloc[-1] / hist.iloc[0] - 1                         # return of the last 12 months
    vol = hist.pct_change().std() * np.sqrt(252)                   # annualized volatility
    SOURCE = f"LIVE data from Yahoo Finance · last price of {hist.index[-1].date()}"
except Exception as e:
    price, ret, vol = (pd.Series({t: v[k] for t, v in SNAPSHOT.items()}) for k in range(3))
    SOURCE = f"OFFLINE snapshot of 2026-10-29 (Yahoo Finance not reachable: {type(e).__name__})"

MARKET = pd.DataFrame({"Company": [INDEX[t][0] for t in INDEX], "Sector": [INDEX[t][1] for t in INDEX],
                       "Price ($)": price[list(INDEX)].values, "Return 12m": ret[list(INDEX)].values,
                       "Volatility": vol[list(INDEX)].values}, index=list(INDEX))
MARKET.index.name = "Ticker"
STOCKS = list(INDEX)
BUDGET, MAX_WEIGHT, MAX_STOCKS, MIN_SECTORS, MAX_VOL = 100_000, 0.25, 6, 3, 0.21


def trade_ticket(shares, show_table=True):
    """Checks a portfolio (dict ticker -> whole shares) against the investor's rules and prints the trade ticket."""
    held = {t: int(round(s)) for t, s in shares.items() if round(s) > 0}
    value = {t: held[t] * MARKET.at[t, "Price ($)"] for t in held}
    total = sum(value.values())
    sectors = {MARKET.at[t, "Sector"] for t in held}
    wvol = sum(value[t] * MARKET.at[t, "Volatility"] for t in held) / total if total else 0
    profit = sum(value[t] * MARKET.at[t, "Return 12m"] for t in held)
    rules = [("Budget", total, "≤", BUDGET), ("Biggest position", max(value.values(), default=0), "≤", MAX_WEIGHT * BUDGET),
             ("Number of stocks", len(held), "≤", MAX_STOCKS), ("Number of sectors", len(sectors), "≥", MIN_SECTORS),
             ("Weighted volatility", wvol, "≤", MAX_VOL)]
    ok = lambda v, s, r: v <= r + 1e-6 if s == "≤" else v >= r - 1e-6
    if show_table:
        display(pd.DataFrame({"Shares": held, "Price ($)": {t: MARKET.at[t, "Price ($)"] for t in held}, "Cost ($)": value})
                .style.format({"Price ($)": "{:,.2f}", "Cost ($)": "{:,.0f}"}))
    for name, v, s, r in rules:
        print(f"{'✔' if ok(v, s, r) else '✘'} {name}: {v:,.2f} {s} {r:,.2f}")
    print(f"Invested ${total:,.0f} · cash left ${BUDGET - total:,.0f} · expected 1-year profit ${profit:,.0f}")


print(SOURCE)
MARKET.style.format({"Price ($)": "{:,.2f}", "Return 12m": "{:.1%}", "Volatility": "{:.1%}"})

### Your tasks
1. **Model (on paper first).** Define the variables (shares bought, and the binary variables you need to count stocks
   and sectors), the objective and **every rule** of the table. Which rules are linear as written? Which one
   needs a trick (hint: multiply the volatility rule by the invested value to remove the division)?
2. **Implement** the model in the function below: the starter code only has the budget and the objective, so it
   puts all the money in one stock. Add the rules one by one and use `trade_ticket` to see which ones are ✘.
3. **Analyze** the solution: which rules are **binding**? How much cash is left, and why? Which sectors did you pick?
4. **What-if analysis:** run the scenarios (cell after the function). How much expected profit does each rule
   cost? How much does the **integrality** cost (compare with the LP relaxation)? What if the budget doubles?
5. **Reflect:** what does the model ignore (risk correlation, transaction costs, taxes)? Would you invest on this advice?

In [ ]:
import pulp

def solve_portfolio(budget=BUDGET, max_weight=None, max_stocks=None, min_sectors=None, max_vol=None, relax=False):
    """Maximizes the expected 12-month profit. Each rule is applied only if its parameter is not None."""
    sectors = sorted(MARKET["Sector"].unique())
    model = pulp.LpProblem("portfolio", pulp.LpMaximize)
    n = {t: model.add_variable(f"n_{t}", lowBound=0, cat="Continuous" if relax else "Integer") for t in STOCKS}    # shares
    y = {t: model.add_variable(f"y_{t}", cat="Binary") for t in STOCKS}                                           # 1 = stock held
    s = {k: model.add_variable(f"s_{k}", cat="Binary") for k in sectors}                                          # 1 = sector held
    value = {t: MARKET.at[t, "Price ($)"] * n[t] for t in STOCKS}                                                 # $ invested in t
    model += pulp.lpSum(MARKET.at[t, "Return 12m"] * value[t] for t in STOCKS)                                    # expected profit
    model += pulp.lpSum(value.values()) <= budget, "budget"
    if max_weight is not None:
        pass                                                      # ✍️ each position at most max_weight * budget
    if max_stocks is not None:
        pass                                                      # ✍️ y_t = 1 exactly when you hold shares of t (link both ways), then count the stocks
    if min_sectors is not None:
        pass                                                      # ✍️ link s_k to the stocks of sector k and count the sectors
    if max_vol is not None:
        pass                                                      # ✍️ sum(vol_t * value_t) <= max_vol * sum(value_t)
    model.solve(pulp.COIN_CMD(msg=False))
    return {t: n[t].value() for t in STOCKS}, pulp.value(model.objective)


RULES = dict(max_weight=MAX_WEIGHT, max_stocks=MAX_STOCKS, min_sectors=MIN_SECTORS, max_vol=MAX_VOL)
shares, profit = solve_portfolio(**RULES)
trade_ticket(shares)

In [ ]:
# What-if analysis: run it after your model works (each row re-solves the model with one change)
scenarios = [("All rules", {}), ("Budget ×2", {"budget": 2 * BUDGET}), ("Max 4 stocks", {"max_stocks": 4}),
             ("No sector rule", {"min_sectors": None}), ("No risk limit", {"max_vol": None}),
             ("No rules (only budget)", {"max_weight": None, "max_stocks": None, "min_sectors": None, "max_vol": None}),
             ("LP relaxation (fractional shares)", {"relax": True})]
rows = []
for name, change in scenarios:
    sh, z = solve_portfolio(**{**RULES, **change})
    cost = sum(sh[t] * MARKET.at[t, "Price ($)"] for t in STOCKS)
    rows.append({"Scenario": name, "Expected profit ($)": z, "Invested ($)": cost,
                 "Stocks": sum(1 for v in sh.values() if v > 1e-6)})
pd.DataFrame(rows).style.format({"Expected profit ($)": "{:,.0f}", "Invested ($)": "{:,.0f}"})

### Reflection (write your answers in a text cell)
* Which rule costs the most expected profit? Is it worth it?
* Compare the integer plan with the LP relaxation: why is the gap so small here, whereas it mattered in the knapsack?
* The expected profit uses the **past** return as if it were certain. How would you change the model to include
  uncertainty (for example, several return scenarios and a worst-case or a minimum-return rule)?